# Variational payoffs and quantum coordination

Companion experiments for *Simulating Game Theory and Strategic Interactions Using Quantum Computing*.

This notebook explores depth-one variational circuits for the Prisoner's Dilemma and Bell-pair protocols for Schelling coordination. Install `python -m pip install -e '.[notebook,verify]'` from the repository root. All displayed values are current exact statevector expectations, evaluated without shot sampling.

In [1]:
from math import pi

import numpy as np
from IPython.display import Markdown, display

from quantum_pd.experiments import (
    classical_match,
    qaoa_outcomes,
    schelling_four_spots,
    schelling_probabilities,
)

## Variational payoff landscape

Prepare each qubit with H, apply pairwise ZZ rotations, and apply an RX mixer. Evaluate the pairwise-summed Prisoner's Dilemma payoffs over a 7 × 7 grid: gamma in $[0,\pi/2]$ and beta in $[0,\pi]$. The table records the best grid point's mean payoff and expected fraction of cooperating players. A grid optimum describes aggregate payoff optimization; individual-player Nash incentives are tested separately.

In [2]:
rows = [
    "| Players | Baseline payoff | Best grid mean payoff | Cooperation fraction |",
    "|---|---|---|---|",
]
for players in (2, 3, 4):
    candidates = [
        qaoa_outcomes(players, gamma, beta)
        for gamma in np.linspace(0, pi / 2, 7)
        for beta in np.linspace(0, pi, 7)
    ]
    best = max(candidates, key=lambda result: result["mean_payoff"])
    rows.append(
        f"| {players} | {players - 1} | {best['mean_payoff']:.6f} | "
        f"{best['cooperation_rate']:.6f} |"
    )
display(Markdown("\n".join(rows)))

| Players | Baseline payoff | Best grid mean payoff | Cooperation fraction |
|---|---|---|---|
| 2 | 1 | 2.466506 | 0.500000 |
| 3 | 2 | 4.593750 | 0.500000 |
| 4 | 3 | 6.782632 | 0.500000 |

## Two-spot coordination

For independent classical choices, the matching probability is $p_Ap_B+(1-p_A)(1-p_B)$. For a Bell pair followed by local RY rotations and measurement, it is $\cos^2((\theta_A-\theta_B)/2)$. The classical probabilities and quantum rotation angles are stated separately.

In [3]:
print("Independent classical choices:")
for a, b in [(0.5, 0.5), (0.7, 0.7), (0.9, 0.9), (0.8, 0.2)]:
    print(f"pA={a}, pB={b}: {classical_match(a, b):.6f}")

rows = ["| Rotation angles | Match probability | With joint inverse |", "|---|---|---|"]
for label, a, b in [
    ("0, 0", 0, 0),
    ("π/6, π/6", pi / 6, pi / 6),
    ("π/6, 0", pi / 6, 0),
    ("π/2, 0", pi / 2, 0),
]:
    raw = schelling_probabilities(a, b)
    decoded = schelling_probabilities(a, b, undo=True)
    match = float(raw[0] + raw[3])
    assert np.isclose(match, np.cos((a - b) / 2) ** 2)
    rows.append(f"| {label} | {match:.6f} | {decoded[0] + decoded[3]:.6f} |")
display(Markdown("\n".join(rows)))

Independent classical choices:
pA=0.5, pB=0.5: 0.500000
pA=0.7, pB=0.7: 0.580000
pA=0.9, pB=0.9: 0.820000
pA=0.8, pB=0.2: 0.320000


| Rotation angles | Match probability | With joint inverse |
|---|---|---|
| 0, 0 | 1.000000 | 1.000000 |
| π/6, π/6 | 1.000000 | 1.000000 |
| π/6, 0 | 0.933013 | 1.000000 |
| π/2, 0 | 0.500000 | 1.000000 |

The inverse variant includes a joint CX operation before measurement, so it is a distinct protocol. Equal rotations preserve matching in the direct Bell-pair protocol. Shared classical randomness can also give perfect matching; the independent-choice baseline above specifies a different resource assumption.

## Four-spot coordination

Two Bell pairs distribute a two-bit register to each player. Compare both registers to determine whether the players selected the same spot. Uniform independent classical choices among four spots match with probability 1/4.

In [4]:
result = schelling_four_spots()
print("Independent classical match probability:", 1 / 4)
print("Bell-pair match probability:", round(result["match_rate"], 12))
print("Unitary gates:", result["unitary_gates"])
print("CX gates included in that total:", result["cx_count"])
print("Measurement operations:", result["measurement_operations"])

Independent classical match probability: 0.25
Bell-pair match probability: 1.0
Unitary gates: 4
CX gates included in that total: 2
Measurement operations: 4


## Reproduce the saved examples

Run `python scripts/reproduce_experiments.py --check` to compare the current calculations with `results/experiments.json`. See [methods](../docs/methods.md) for the circuit definitions and numerical conventions.